# <font color="#F48E16">XAI aplicada al Deep Learning: Oclusión</font>

Material generado por <a href="https://www.linkedin.com/in/christian-oliva-moya-ingeniero/">Christian Oliva</a>. Cualquier duda, sugerencia o errata, no duden en contactar.

**Versión 1.0** - 28 de abril de 2026

In [ ]:
import tensorflow as tf

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from tqdm import tqdm

## <font color="#F48E16">Datos</font>

En este notebook se muestra la implementación manual del algoritmo de relevancia por oclusión:

Para ello, se van a utilizar diferentes modelos sencillos de SKLearn sobre un dataset que todos conocemos: **California Housing**

<hr>

En el notebook se desarrolla el código por completo según una fase sencilla de preprocesamiento de los datos, el entrenamiento de algunos modelos y la explicabilidad utilizando los diferentes algoritmos.

### Descarga de datos de SKLearn

In [ ]:
# Download latest version
housing = fetch_california_housing()
X = housing.data
y = housing.target

In [ ]:
columnas = housing.feature_names
print(columnas)

### Primer vistazo de los datos

In [ ]:
plt.figure(figsize=(15, 6))
for i in range(len(housing.feature_names)):
  plt.subplot(2, 4, i+1)
  plt.title(housing.feature_names[i])
  plt.hist(X[:, i], bins=21)
plt.show()

#### TODO JUNTO

In [ ]:
X[:, 0] = np.log(X[:, 0]) # MedInc
X[:, 2] = np.log(X[:, 2]) # AveRooms
X[:, 3] = np.log(X[:, 3]) # AveBedrms
X[:, 4] = np.log(X[:, 4]) # Population
X[:, 5] = np.log(X[:, 5]) # AveOccup

In [ ]:
plt.figure(figsize=(15, 6))
for i in range(len(housing.feature_names)):
  plt.subplot(2, 4, i+1)
  plt.title(housing.feature_names[i])
  plt.hist(X[:, i], bins=21)
plt.show()

### Separación en TRAIN-TEST y normalización

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
means = X_train_raw.mean()
stds = X_train_raw.std()
X_train = (X_train_raw-means) / stds
X_test = (X_test_raw-means) / stds

## <font color="#F48E16">Modelos</font>

### Regresión Lineal (modelo lineal)

In [ ]:
rl = LinearRegression()
rl.fit(X_train, y_train)
rl.score(X_test, y_test)

### SVM

In [ ]:
svc = SVR(kernel="rbf", C=1.0, gamma="scale", max_iter=2000)
svc.fit(X_train, y_train)
svc.score(X_test, y_test)

### MLP

In [ ]:
red = MLPRegressor(hidden_layer_sizes=(100,), max_iter=2000)
red.fit(X_train, y_train)
red.score(X_test, y_test)

## <font color="#F48E16">Explicabilidad</font>

**IMPORTANTE**: Por simplicidad para los algoritmos más pesados, vamos a utilizar solo un trocito de X_train como si fuese nuestro dataset completo.

In [ ]:
X_train_small = X_train[:100]
y_train_small = y_train[:100]

In [ ]:
def importancia_permutacion(X, y, model, N=10):
    Rx = np.zeros(X.shape[1])

    ##################################
    # TO-DO Implementa el algoritmo de importancia por permutación
    ##################################

    base_score = model.score(X, y)
    num_atributos = X.shape[1]

    for i in range(num_atributos):
      for j in range(N):
        X_perm = X.copy()
        X_perm[:, i] = np.random.permutation(X_perm[:, i])
        Rx[i] += model.score(X_perm, y)

    Rx = base_score - Rx / N

    return Rx

In [ ]:
Rx_perm_rl = importancia_permutacion(X_train_small, y_train_small, rl, N=10)
Rx_perm_svc = importancia_permutacion(X_train_small, y_train_small, svc, N=10)
Rx_perm_red = importancia_permutacion(X_train_small, y_train_small, red, N=10)

In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(columnas, Rx_perm_rl)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.axhline(0.0, color="red")
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(columnas, Rx_perm_svc)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(columnas, Rx_perm_red)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.show()

In [ ]:
x = np.arange(len(columnas))
width = 0.25

plt.figure(figsize=(10, 3))
plt.bar(x - width, np.abs(Rx_perm_rl), width, label="Regresión Logística")
plt.bar(x, np.abs(Rx_perm_svc), width, label="SVC")
plt.bar(x + width, np.abs(Rx_perm_red), width, label="Red Neuronal")
plt.xticks(x, columnas, rotation=90)
plt.ylabel("Importancia por Permutación")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### Relevancia por oclusión [Método local y global]

La relevancia por oclusión, en general, se utiliza como técnica de explicabilidad de modelos que procesan imágenes, pero, aunque es una técnica que veremos más adelante aplicada a dicha tarea, también puede ser utilizada en problemas tabulares. La relevancia por oclusión es un método híbrido, que puede ser aplicado de forma tanto local (para una observación concreta) como global (para todo el dataset), que consiste en anular una región del espacio de atributos de entrada.

El término anular consiste en reemplazar un atributo $i$-ésimo por un valor "inteligente", por ejemplo:

- Un valor fijo 0

- La media del atributo

- La moda del atributo

- Un valor imputado mediante un algoritmo más complejo

La relevancia por oclusión global para datos tabulares se calcula, entonces, de la siguiente manera:

$$R_i = M(X) - M(X'_i)$$

donde $M(X)$ representa una métrica de rendimiento del modelo $M$ al procesar el dataset $X$ y $M(X'_i)$ representa la misma métrica de rendimiento al procesar el dataset con el atributo $i$-ésimo anulado.

Sin embargo, si se busca la oclusión a nivel local, se calcularía de la siguiente manera:

$$R_i(x) = f(x) - f(x'_i)$$

donde $f(x)$ es la predicción del modelo para el dato de entrada $x$ y $f(x'_i)$ es la predicción del mismo modelo para el dato de entrada $x$ con el atributo $i$-ésimo anulado.

**¿Por qué se sigue esta estrategia?**

Esta estrategia se basa en una idea parecida a la importancia por permutación, donde se evalúa el modelo después de modificar un atributo de entrada del dataset.

En el siguiente código se va a utilizar el valor fijo 0 porque es la media de los atributos de entrada, ya que están normalizados.

In [ ]:
def relevancia_oclusion(X, y, model, verbose=True):
  if X.ndim == 1: # 1 solo dato, lo ponemos en formato matricial
    X = X[None, :]
    pred = model.predict(X)
  else:
    score = model.score(X, y)

  num_atributos = X.shape[-1]

  Rx = np.zeros(num_atributos)
  ##################################
  # TO-DO Implementa la relevancia por oclusión tanto global como local
  ##################################

  return Rx

**Relevancia global**

In [ ]:
Rx_oclu_rl = relevancia_oclusion(X_train_small, y_train_small, rl)
Rx_oclu_svc = relevancia_oclusion(X_train_small, y_train_small, svc)
Rx_oclu_red = relevancia_oclusion(X_train_small, y_train_small, red)

In [ ]:
x = np.arange(len(columnas))
width = 0.3

plt.figure(figsize=(10, 3))
plt.bar(x - width/2, Rx_perm_rl / Rx_perm_rl.max(), width, label="Importancia por permutación")
plt.bar(x + width/2, Rx_oclu_rl / Rx_oclu_rl.max(), width, label="Relevancia por oclusión")
plt.xticks(x, columnas, rotation=90)
plt.title("Regresión Logística")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
x = np.arange(len(columnas))
width = 0.3

plt.figure(figsize=(10, 3))
plt.bar(x - width/2, Rx_perm_svc, width, label="Importancia por permutación")
plt.bar(x + width/2, Rx_oclu_svc, width, label="Relevancia por oclusión")
plt.xticks(x, columnas, rotation=90)
plt.title("SVC")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
x = np.arange(len(columnas))
width = 0.3

plt.figure(figsize=(10, 3))
plt.bar(x - width/2, Rx_perm_red / Rx_perm_red.max(), width, label="Importancia por permutación")
plt.bar(x + width/2, Rx_oclu_red / Rx_oclu_red.max(), width, label="Relevancia por oclusión")
plt.xticks(x, columnas, rotation=90)
plt.title("Red Neuronal")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

Como hemos podido ver, **dos métodos diferentes de explicabilidad dan dos resultados también diferentes**.

**Relevancia local**

Ahora vamos a explicar un dato particular.

In [ ]:
item = 0

In [ ]:
Rx_oclu_local_rl = relevancia_oclusion(X_train_small[item], y_train_small[item], rl)
Rx_oclu_local_svc = relevancia_oclusion(X_train_small[item], y_train_small[item], svc)
Rx_oclu_local_red = relevancia_oclusion(X_train_small[item], y_train_small[item], red)

In [ ]:
x = np.arange(len(columnas))
width = 0.3

plt.figure(figsize=(10, 3))
plt.bar(x - width/2,  Rx_oclu_rl / Rx_oclu_rl.max(), width, label="Global")
plt.bar(x + width/2, Rx_oclu_local_rl / Rx_oclu_local_rl.max(), width, label="Local item="+str(item))
plt.xticks(x, columnas, rotation=90)
plt.title("Relevancia por oclusión con Regresión Lineal")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

¿Por qué con el item=0 pasa que loan_percent_income, por ejemplo, es negativo?

Primero vemos el dato original y su predicción:

In [ ]:
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.bar(columnas, X_train_small[item])
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.subplot(1, 2, 2)
plt.plot([0], rl.predict(X_train_small[item][None, :])[0], "o")
plt.xticks([0], ["Pred"])
plt.title("Valor real: " + str(y_train_small[item]))
plt.grid()
plt.show()

Vamos a anular **MedInc**, es decir, hacerlo más pequeño.

In [ ]:
X_null = X_train_small[item].copy()
X_null = X_null[None, :]
X_null[:, 0] = 0

plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.bar(columnas, X_null[0])
plt.bar(columnas, X_train_small[item], alpha=0.2)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.subplot(1, 2, 2)
plt.plot([0], rl.predict(X_train_small[item][None, :])[0], "o", label="Original")
plt.plot([0], rl.predict(X_null)[0], 'o', label="Anulado")
plt.xticks([0], ["Pred"])
plt.grid()
plt.legend()
plt.show()

# ¿Tiene sentido?
# Si anulas un atributo cuya explicabilidad con oclusión es positiva,
# la predicción DISMINUYE

Vamos a anular ahora **Latitude**, es decir, bajarlo.

In [ ]:
X_null = X_train_small[item].copy()
X_null = X_null[None, :]
X_null[:, -2] = 0

plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
plt.bar(columnas, X_null[0])
plt.bar(columnas, X_train_small[item], alpha=0.2)
plt.xticks(rotation=90)
plt.grid(alpha=0.2)
plt.subplot(1, 2, 2)
plt.plot([0], rl.predict(X_train_small[item][None, :])[0], "o", label="Original")
plt.plot([0], rl.predict(X_null)[0], 'o', label="Anulado")
plt.xticks([0], ["Pred"])
plt.grid()
plt.legend()
plt.show()

# ¿Tiene sentido?
# Si anulas un atributo cuya explicabilidad con oclusión es negativa,
# la predicción AUMENTA